# Hospital Readmission Risk Prediction & Healthcare Analytics

**Note:** The included dataset is synthetic and is intended for learning/portfolio demonstration, not clinical decision-making.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/healthcare_readmission.csv")
df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.isna().sum())
print(df["readmitted_30_days"].value_counts(normalize=True))

In [ ]:
print(df.groupby("age_group")["readmitted_30_days"].mean().sort_values(ascending=False))
print(df.groupby("diagnosis")["readmitted_30_days"].mean().sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x="diagnosis", y="readmitted_30_days")
plt.xticks(rotation=30)
plt.ylabel("30-day readmission rate")
plt.title("Readmission Rate by Diagnosis")
plt.tight_layout()
plt.show()

## Modeling

Use one-hot encoding for categorical variables and compare Logistic Regression, Random Forest and XGBoost. Evaluate using precision, recall, F1 and ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

X = df.drop(columns=["patient_id","readmitted_30_days"])
y = df["readmitted_30_days"]

cat = X.select_dtypes(include="object").columns
num = X.select_dtypes(exclude="object").columns

pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")
}

for name, model in models.items():
    pipe = Pipeline([("preprocess", pre), ("model", model)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:,1]
    pred = (p >= .5).astype(int)
    print("\n", name)
    print(classification_report(y_test, pred))
    print("ROC-AUC:", roc_auc_score(y_test, p))

In [ ]:
# Optional XGBoost model
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=.05,
    subsample=.85, colsample_bytree=.85, eval_metric="logloss",
    random_state=42
)
xgb_pipe = Pipeline([("preprocess", pre), ("model", xgb)])
xgb_pipe.fit(X_train, y_train)
p = xgb_pipe.predict_proba(X_test)[:,1]
pred = (p >= .5).astype(int)
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, p))